# Asyncio — асинхронное программирование в Python

## Зачем нужна асинхронность

Обычный (синхронный) код выполняется строго последовательно: пока одна операция не завершится,
следующая не начнётся. Это критично для **I/O-bound** задач — сетевые запросы, чтение файлов,
обращения к базе данных — где программа большую часть времени просто *ждёт* ответа, а не считает.

Асинхронное программирование позволяет во время такого ожидания переключаться на другую задачу,
не создавая отдельные потоки или процессы. `asyncio` — стандартная библиотека Python для этого:
она работает в одном потоке и использует **event loop** (цикл событий), который переключается
между задачами в моменты ожидания (`await`).

Важно: `asyncio` не даёт параллелизма для CPU-bound кода (из-за GIL). Она ускоряет именно
задачи, упирающиеся в ожидание ввода-вывода.

## Синхронный пример: имитация сетевых запросов

In [1]:
import time

def fetch_sync(name: str, delay: float) -> str:
    print(f'{name}: начало запроса')
    time.sleep(delay)  # имитация ожидания ответа сервера
    print(f'{name}: запрос завершён')
    return f'{name}: результат'

start = time.perf_counter()

results = [
    fetch_sync('Запрос A', 1),
    fetch_sync('Запрос B', 1),
    fetch_sync('Запрос C', 1),
]

print(results)
print(f'Общее время: {time.perf_counter() - start:.2f} сек')

Запрос A: начало запроса


Запрос A: запрос завершён
Запрос B: начало запроса


Запрос B: запрос завершён
Запрос C: начало запроса


Запрос C: запрос завершён
['Запрос A: результат', 'Запрос B: результат', 'Запрос C: результат']
Общее время: 3.00 сек


Три «запроса» по 1 секунде выполнились строго друг за другом — итого около 3 секунд,
хотя всё это время программа просто ждала.

## Корутины: `async def` и `await`

In [2]:
import asyncio

async def fetch_async(name: str, delay: float) -> str:
    print(f'{name}: начало запроса')
    await asyncio.sleep(delay)  # не блокирует поток, отдаёт управление event loop
    print(f'{name}: запрос завершён')
    return f'{name}: результат'

# Простой вызов НЕ запускает корутину — создаётся только объект-корутина
coro = fetch_async('Запрос A', 1)
print(coro)
coro.close()  # закрываем, чтобы не было предупреждения о незавершённой корутине

<coroutine object fetch_async at 0x0000029097A925A0>


Вызов `async`-функции не выполняет её тело — он создаёт **объект-корутину**.
Чтобы код внутри реально начал работать, корутину нужно передать в event loop:
через `await` (внутри другой корутины) или через `asyncio.run(...)` (снаружи).

> **Про Jupyter:** сам ноутбук уже выполняется внутри своего event loop, поэтому обычный
> `asyncio.run(...)` в нём упадёт с `RuntimeError: asyncio.run() cannot be called from a
> running event loop`. Библиотека `nest_asyncio` разрешает вложенные циклы событий — это
> нужно только для демонстрации в ноутбуке. В обычном `.py`-скрипте `asyncio.run(main())`
> работает без каких-либо дополнительных действий.

In [3]:
import nest_asyncio

nest_asyncio.apply()

In [4]:
async def main():
    result = await fetch_async('Запрос A', 1)
    print(result)

asyncio.run(main())

Запрос A: начало запроса


Запрос A: запрос завершён
Запрос A: результат


## Последовательное ожидание `await` не даёт параллелизма

In [5]:
async def main_sequential():
    start = time.perf_counter()

    r1 = await fetch_async('Запрос A', 1)
    r2 = await fetch_async('Запрос B', 1)
    r3 = await fetch_async('Запрос C', 1)

    print([r1, r2, r3])
    print(f'Общее время: {time.perf_counter() - start:.2f} сек')

asyncio.run(main_sequential())

Запрос A: начало запроса


Запрос A: запрос завершён
Запрос B: начало запроса


Запрос B: запрос завершён
Запрос C: начало запроса


Запрос C: запрос завершён
['Запрос A: результат', 'Запрос B: результат', 'Запрос C: результат']
Общее время: 3.01 сек


Каждый `await` дожидается завершения корутины, прежде чем перейти к следующей строке —
поведение получилось таким же последовательным, как в синхронном варианте (~3 секунды).
Чтобы получить настоящую конкурентность, задачи нужно запускать одновременно.

## Конкурентное выполнение: `asyncio.gather`

In [6]:
async def main_gather():
    start = time.perf_counter()

    results = await asyncio.gather(
        fetch_async('Запрос A', 1),
        fetch_async('Запрос B', 1),
        fetch_async('Запрос C', 1),
    )

    print(results)
    print(f'Общее время: {time.perf_counter() - start:.2f} сек')

asyncio.run(main_gather())

Запрос A: начало запроса
Запрос B: начало запроса
Запрос C: начало запроса


Запрос A: запрос завершён
Запрос B: запрос завершён
Запрос C: запрос завершён
['Запрос A: результат', 'Запрос B: результат', 'Запрос C: результат']
Общее время: 1.00 сек


`asyncio.gather` планирует все переданные корутины сразу и ждёт их совместного завершения.
Пока одна «ждёт ответа сервера» (`await asyncio.sleep`), event loop переключается на другие —
поэтому общее время близко не к сумме, а к максимуму из задержек (~1 секунда вместо 3).

## `asyncio.create_task`: планирование задачи заранее

In [7]:
async def main_tasks():
    start = time.perf_counter()

    # create_task сразу планирует выполнение в event loop,
    # ещё до того, как мы дойдём до await
    task_a = asyncio.create_task(fetch_async('Задача A', 1))
    task_b = asyncio.create_task(fetch_async('Задача B', 1))

    print('Задачи запланированы, можно делать что-то ещё до await')

    result_a = await task_a
    result_b = await task_b

    print([result_a, result_b])
    print(f'Общее время: {time.perf_counter() - start:.2f} сек')

asyncio.run(main_tasks())

Задачи запланированы, можно делать что-то ещё до await
Задача A: начало запроса
Задача B: начало запроса


Задача A: запрос завершён
Задача B: запрос завершён
['Задача A: результат', 'Задача B: результат']
Общее время: 1.00 сек


В отличие от простого `await coro`, `create_task` возвращает объект `Task` и запускает
корутину в фоне немедленно — она начинает выполняться, даже если результат ещё не запрошен
через `await`. Это удобно, когда задачи нужно запустить не в один момент кода.

## Частая ошибка: блокирующий код внутри корутины

In [8]:
async def fetch_blocking(name: str, delay: float) -> str:
    print(f'{name}: начало запроса')
    time.sleep(delay)  # ОШИБКА: блокирует весь event loop, а не только эту корутину
    print(f'{name}: запрос завершён')
    return f'{name}: результат'

async def main_blocking_mistake():
    start = time.perf_counter()

    await asyncio.gather(
        fetch_blocking('Запрос A', 1),
        fetch_blocking('Запрос B', 1),
    )

    print(f'Общее время: {time.perf_counter() - start:.2f} сек')

asyncio.run(main_blocking_mistake())

Запрос A: начало запроса


Запрос A: запрос завершён
Запрос B: начало запроса


Запрос B: запрос завершён
Общее время: 2.00 сек


Несмотря на `asyncio.gather`, задачи снова выполнились последовательно (~2 секунды).
`time.sleep` — синхронная, блокирующая функция: она замораживает весь поток целиком,
включая event loop, так что переключиться на другую задачу невозможно. Внутри корутин
всегда нужно использовать асинхронные аналоги (`asyncio.sleep`, асинхронные HTTP-клиенты
вроде `aiohttp`, асинхронные драйверы БД и т.д.).

## Таймауты: `asyncio.wait_for`

In [9]:
async def slow_operation():
    await asyncio.sleep(2)
    return 'готово'

async def main_timeout():
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=1)
        print(result)
    except asyncio.TimeoutError:
        print('Операция не уложилась в 1 секунду — отменена по таймауту')

asyncio.run(main_timeout())

Операция не уложилась в 1 секунду — отменена по таймауту


`asyncio.wait_for` оборачивает корутину и, если она не успевает завершиться за отведённое
время, отменяет её и выбрасывает `asyncio.TimeoutError`. Это защищает программу от
«зависания» на операциях, которые могут выполняться неограниченно долго.

## Обработка исключений в конкурентных задачах

In [10]:
async def fetch_may_fail(name: str, should_fail: bool):
    await asyncio.sleep(0.3)
    if should_fail:
        raise ValueError(f'{name}: сервер вернул ошибку')
    return f'{name}: результат'

async def main_exceptions():
    # По умолчанию gather прерывается на первом же исключении
    try:
        await asyncio.gather(
            fetch_may_fail('Запрос A', False),
            fetch_may_fail('Запрос B', True),
        )
    except ValueError as e:
        print(f'Поймано исключение: {e}')

    # return_exceptions=True собирает исключения как обычные результаты,
    # не прерывая остальные задачи
    results = await asyncio.gather(
        fetch_may_fail('Запрос C', False),
        fetch_may_fail('Запрос D', True),
        return_exceptions=True,
    )
    print(results)

asyncio.run(main_exceptions())

Поймано исключение: Запрос B: сервер вернул ошибку


['Запрос C: результат', ValueError('Запрос D: сервер вернул ошибку')]


## Отмена задач

In [11]:
async def long_task():
    try:
        print('Длинная задача: начало')
        await asyncio.sleep(5)
        print('Длинная задача: это не должно напечататься')
    except asyncio.CancelledError:
        print('Длинная задача: отменена, выполняем очистку ресурсов')
        raise  # важно пробросить исключение дальше

async def main_cancel():
    task = asyncio.create_task(long_task())
    await asyncio.sleep(0.5)  # даём задаче немного поработать
    task.cancel()

    try:
        await task
    except asyncio.CancelledError:
        print('main: задача была отменена')

asyncio.run(main_cancel())

Длинная задача: начало


Длинная задача: отменена, выполняем очистку ресурсов
main: задача была отменена


`task.cancel()` посылает в корутину `CancelledError` в точке следующего `await`.
Внутри корутины это исключение можно перехватить, чтобы освободить ресурсы (закрыть
соединение, файл и т.п.), но обычно его нужно пробрасывать дальше — иначе задача будет
считаться завершившейся успешно.

## Producer/consumer: `asyncio.Queue`

In [12]:
async def producer(queue: asyncio.Queue, count: int):
    for i in range(count):
        await asyncio.sleep(0.2)
        item = f'item-{i}'
        await queue.put(item)
        print(f'Producer: положил {item} в очередь')
    await queue.put(None)  # сигнал об окончании работы

async def consumer(queue: asyncio.Queue):
    while True:
        item = await queue.get()
        if item is None:
            break
        print(f'Consumer: обработал {item}')

async def main_queue():
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue, 3),
        consumer(queue),
    )

asyncio.run(main_queue())

Producer: положил item-0 в очередь
Consumer: обработал item-0


Producer: положил item-1 в очередь
Consumer: обработал item-1
Producer: положил item-2 в очередь


Consumer: обработал item-2


`asyncio.Queue` — асинхронная очередь для обмена данными между корутинами: `producer`
кладёт элементы через `await queue.put(...)`, а `consumer` забирает их через
`await queue.get()`, при необходимости дожидаясь появления нового элемента.

## Защита общего ресурса: `asyncio.Lock`

In [13]:
balance = 100

async def withdraw(name: str, amount: int, lock: asyncio.Lock):
    async with lock:
        global balance
        print(f'{name}: проверяю баланс ({balance})')
        await asyncio.sleep(0.1)  # имитация обращения к БД
        if balance >= amount:
            balance -= amount
            print(f'{name}: снял {amount}, остаток {balance}')
        else:
            print(f'{name}: недостаточно средств')

async def main_lock():
    lock = asyncio.Lock()
    await asyncio.gather(
        withdraw('Клиент A', 80, lock),
        withdraw('Клиент B', 80, lock),
    )

asyncio.run(main_lock())

Клиент A: проверяю баланс (100)
Клиент A: снял 80, остаток 20
Клиент B: проверяю баланс (20)


Клиент B: недостаточно средств


Без `Lock` обе корутины могли бы одновременно прочитать одинаковый `balance` до того,
как кто-то из них его изменит (race condition), и оба списания могли бы пройти. `async with
lock` гарантирует, что в блок кода одновременно попадает только одна задача.

## Итоги

- `asyncio` даёт конкурентность в одном потоке за счёт переключения на точках `await`
- `async def` создаёт корутину; сам вызов ничего не выполняет — нужен `await` или `asyncio.run`
- Последовательные `await` не дают ускорения — для конкурентности нужны `gather` / `create_task`
- Внутри корутин нельзя использовать блокирующие вызовы (`time.sleep`, синхронные I/O) — они
  замораживают весь event loop
- `wait_for` — таймауты, `gather(..., return_exceptions=True)` — сбор ошибок без обрыва
  остальных задач, `task.cancel()` — явная отмена
- `asyncio.Queue` и `asyncio.Lock` — стандартные примитивы координации между корутинами